# Deploying and Operating Machine Learning Models

---
## Guided Walkthrough: From a `scikit-learn` Pipeline to a Local MLflow Endpoint

In the previous lesson, you created reproducible machine learning workflows with scikit-learn pipelines. In this walkthrough, you will take an existing pipeline beyond the modelling notebook and prepare it for use by another system.

### About

This notebook demonstrates a simplified MLOps workflow using MLflow. You will track an experiment, package a complete preprocessing-and-model pipeline, load the logged artifact, serve it through a local REST endpoint, and test the endpoint with valid and invalid requests.

### Learning objectives

- Track parameters, metrics, tags, data lineage, and model artifacts with MLflow.
- Package a complete scikit-learn pipeline with an input example and model signature.
- Load a logged model and confirm that it can reproduce predictions.
- Serve and test the model as a local REST endpoint.


### Notebook guide

1. Imports and local setup
2. Load the existing heart-disease data
3. Fit the selected scikit-learn pipeline
4. Track and package the model with MLflow
5. Inspect the experiment in the MLflow UI
6. Load the logged model
7. Serve the model locally
8. Test valid and invalid requests
9. Connect the local demonstration to production monitoring


## Setup

Run the installation cell only when the lesson environment has not already been prepared. Restart the kernel after installing packages.

```python
%pip install -r requirements.txt
```


In [ ]:
from __future__ import annotations  # lets type hints (e.g. -> dict) be used without affecting how the code runs

import json
import os
import sys
import time
from pathlib import Path

import atexit        # ensures background MLflow processes are shut down when the notebook/kernel closes
import socket         # used to check whether a port is already in use before starting a server
import subprocess     # launches the MLflow UI and model server as separate background processes

import numpy as np
import pandas as pd
import requests
import sklearn
import mlflow          # tracks experiments and packages/loads models
import mlflow.sklearn  # saves a scikit-learn pipeline in a format MLflow can log, load, and serve

from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="mlflow")
# Suppresses MLflow's schema-inference warnings (e.g. about integer columns and dataset source resolution) so the demo output stays readable.
# These are informational only and don't affect this lesson.

print(f"Python: {sys.version.split()[0]}")
print(f"MLflow: {mlflow.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

## Setup: find the data and point MLflow at a local store

Before we can track anything, MLflow needs somewhere to save runs, and we need to be able to find the data set regardless of which folder the notebook is launched from.

- `find_data_file()` searches a few likely locations so this notebook works whether you run it from the lesson folder or a `notebooks/` subfolder.
- `WORK_DIR`, `MLRUNS_DIR`, and `LOG_DIR` set up a local folder called `mlops_artifacts` with sub-folders of `mlruns` and `logs`to hold everything MLflow creates during this session.
- `TRACKING_URI` tells MLflow to store experiment data as files on this machine, rather than sending it to a remote server. This keeps today's exercise self-contained. In a real team setting, this URI would usually point to a shared tracking server instead.

In [ ]:
def find_data_file(filename: str) -> Path:
    """Find the course data whether the notebook is run from the lesson folder or a notebooks folder."""
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "data" / filename,
        Path.cwd().parent / "data" / filename,
        Path("/mnt/data") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")

DATA_PATH = find_data_file("SAheart.data")
WORK_DIR = (Path.cwd() / "mlops_artifacts").resolve()
MLRUNS_DIR = WORK_DIR / "mlruns"
LOG_DIR = WORK_DIR / "logs"

MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# A local file-based tracking store keeps this lesson self-contained.
TRACKING_URI = MLRUNS_DIR.as_uri()
mlflow.set_tracking_uri(TRACKING_URI)

print(f"Data: {DATA_PATH}")
print(f"Tracking store: {TRACKING_URI}")

## Setup: helper functions for running MLflow's servers in the background

Later in this notebook we'll start two servers — the MLflow UI and the model endpoint — without leaving the notebook. These helper functions manage that:

- `port_is_open()` checks whether something is already running on a given port, so we don't accidentally try to start a second server on top of one that's already there.
- `wait_for_url()` polls a URL until it responds, so the notebook waits for a server to finish starting up before trying to use it.
- `start_background_process()` launches a command (like `mlflow ui`) as a background process and keeps a log of its output, so it doesn't block the notebook while it runs.
- `cleanup_background_processes()` shuts everything down cleanly. `atexit.register(...)` ensures this runs automatically when the kernel closes, so servers don't keep running after you've finished the notebook.

You won't need to call these directly; they're used by the setup cells further down.

In [ ]:
_BACKGROUND_PROCESSES = []


def port_is_open(port: int, host: str = "127.0.0.1") -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.25)
        return sock.connect_ex((host, port)) == 0


def wait_for_url(url: str, timeout: int = 30) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            response = requests.get(url, timeout=1)
            if response.status_code < 500:
                return True
        except requests.RequestException:
            pass
        time.sleep(0.5)
    return False


def start_background_process(command: list[str], log_name: str, env: dict | None = None):
    log_path = LOG_DIR / log_name
    log_file = open(log_path, "w", encoding="utf-8")
    process = subprocess.Popen(
        command,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=env,
        text=True,
    )
    _BACKGROUND_PROCESSES.append((process, log_file))
    return process, log_path


def cleanup_background_processes():
    for process, log_file in _BACKGROUND_PROCESSES:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
        log_file.close()


atexit.register(cleanup_background_processes)


# 1. Load and prepare the data

This is the same South African heart-disease dataset used in the **Gridsearch and Pipelines** lesson. The target, `chd`, indicates whether coronary heart disease was present.

We will keep the earlier preprocessing choice: convert `famhist` to a numeric indicator and keep scaling inside the pipeline.

In [ ]:
heart = pd.read_csv(DATA_PATH, skipinitialspace=True)
heart["famhist"] = (heart["famhist"] == "Present").astype(int)

X = heart.drop(columns=["row.names", "chd"])
y = heart["chd"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=2022,
    stratify=y,
)

print(f"Rows: {len(heart):,}")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
X_train.head()

In [ ]:
heart.head()

# 2. Fit the selected pipeline

The pipeline packages preprocessing and prediction as one object:

**raw feature data -> StandardScaler -> LogisticRegression -> prediction**

This is important for deployment. The system making a prediction should not have to remember or re-create the training-time scaling steps.


In [ ]:
baseline_model = Pipeline([
    ("sc", StandardScaler()),
    (
        "lr",
        LogisticRegression(
            C=0.123,
            penalty="l1",
            solver="liblinear",
            max_iter=2_000,
            random_state=2022,
        ),
    ),
])

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)
baseline_probabilities = baseline_model.predict_proba(X_test)[:, 1]

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_roc_auc = roc_auc_score(y_test, baseline_probabilities)

print(f"Accuracy: {baseline_accuracy:.3f}")
print(f"ROC AUC:  {baseline_roc_auc:.3f}")

The model has been developed and evaluated. It is still only a Python object in this notebook.

The next steps make the run traceable and the model portable.


# 3. Track the experiment and package the model

An MLflow **experiment** groups related runs. A **run** records one attempt, including its configuration, results, metadata, and artifacts.

We will log:

- **Parameters**: model type, `C`, penalty, solver, and random seed.
- **Metrics**: accuracy and ROC AUC.
- **Tags**: purpose, owner, and deployment pattern.
- **Data lineage**: the dataset used for the run.
- **Model artifact**: the complete fitted pipeline.
- **Model contract**: an input example and inferred input/output signature.


Here's how each part of the code below maps to the list above:

- `set_experiment` → groups this run with others on the same problem.
- `input_example` and `signature` → the **model contract**.
- `mlflow.data.from_pandas(...)` + `log_input(...)` → **data lineage**.
- `log_params` → **parameters**.
- `log_metrics` → **metrics**.
- `set_tags` → **tags**.
- `mlflow.sklearn.log_model(...)` → the **model artifact**, packaged with its contract, so it can be loaded and served outside this notebook.

The `run_id` and `model_uri` printed at the end are how we'll find and load this exact model in the next steps.

In [ ]:
EXPERIMENT_NAME = "heart-disease-mlops"
mlflow.set_experiment(EXPERIMENT_NAME)

input_example = X_test.iloc[:3].copy()
example_predictions = baseline_model.predict(input_example)
signature = infer_signature(input_example, example_predictions)

dataset = mlflow.data.from_pandas(
    heart.drop(columns=["row.names"]),
    source=str(DATA_PATH),
    targets="chd",
    name="SAheart",
)

with mlflow.start_run(run_name="baseline-logistic-pipeline") as run:
    mlflow.log_params({
        "model_type": "LogisticRegression",
        "scaler": "StandardScaler",
        "C": 0.123,
        "penalty": "l1",
        "solver": "liblinear",
        "random_state": 2022,
    })
    mlflow.log_metrics({
        "accuracy": baseline_accuracy,
        "roc_auc": baseline_roc_auc,
    })
    mlflow.set_tags({
        "lesson": "Deploying and Operating Machine Learning Models",
        "purpose": "coronary heart disease risk classification",
        "owner": "data-science-learner",
        "deployment_pattern": "local-online-endpoint",
    })
    mlflow.log_input(dataset, context="training")

    model_info = mlflow.sklearn.log_model(
        sk_model=baseline_model,
        name="heart_disease_pipeline",
        signature=signature,
        input_example=input_example,
    )

    run_id = run.info.run_id
    model_uri = model_info.model_uri

print(f"Run ID: {run_id}")
print(f"Model URI: {model_uri}")


## What did MLflow add?

The model is no longer an unnamed object that exists only in notebook memory. We now have:

- A unique run ID.
- A record of how the model was configured.
- Metrics for comparing it with other runs.
- A stored model artifact.
- A schema describing the expected inputs and outputs.
- A model URI that another process can use to load or serve it.


# 4. Inspect the run in the MLflow UI

The next cell starts the local MLflow UI on port `5000`.

In the UI, locate:

1. The `heart-disease-mlops` experiment.
2. The `baseline-logistic-pipeline` run.
3. The parameters and metrics.
4. The data set input.
5. The logged model, signature, and input example.

In [ ]:
UI_PORT = 5000
UI_URL = f"http://127.0.0.1:{UI_PORT}"

venv_bin = str(Path(sys.executable).parent)

env = os.environ.copy()
env["MLFLOW_TRACKING_URI"] = TRACKING_URI
env["PATH"] = venv_bin + os.pathsep + env.get("PATH", "")

if port_is_open(UI_PORT):
    print(f"Port {UI_PORT} is already in use. Open {UI_URL} and check whether MLflow is running there.")
else:
    ui_command = [
        sys.executable,
        "-m",
        "mlflow",
        "ui",
        "--backend-store-uri",
        TRACKING_URI,
        "--host",
        "127.0.0.1",
        "--port",
        str(UI_PORT),
    ]
    ui_process, ui_log = start_background_process(ui_command, "mlflow-ui.log", env=env)
    if wait_for_url(UI_URL, timeout=30):
        print(f"MLflow UI is running at {UI_URL}")
    else:
        print(f"MLflow UI did not start. Inspect the log: {ui_log}")

Open: **http://127.0.0.1:5000**

**Discuss**:

- Which details would help you reproduce the model six months from now?
- Which details would help you decide whether this version should be deployed?
- What information is still missing from a real production release?

<details>
<summary>Sample discussion points (click to expand)</summary>

**Reproduce six months from now:**
- The parameters (`C`, `penalty`, `solver`, `random_state`) — enough to refit the same model.
- The dataset lineage (`log_input`) — which data version produced this run.
- Library versions (`sklearn`, `mlflow`) — pipeline behavior can shift across versions.

**Decide whether to deploy:**
- The metrics (`accuracy`, `roc_auc`) compared against the current production model.
- The tags — confirms this run was built for the right use case.
- The signature and input example — confirms the model's contract is correct.

**Missing from a real production release:**
- A model registry entry or version stage (staging/production).
- A real owner and escalation path.
- An approval/review record.
- Monitoring and alerting.
- A rollback plan.

</details>

# 5. Load the logged model

Before serving the model, load it from MLflow and verify that it still works.

We will load the generic MLflow `pyfunc` representation. This provides a consistent prediction interface across different model frameworks.

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_uri)

loaded_predictions = loaded_model.predict(input_example)

comparison = input_example.copy()
comparison["original_prediction"] = example_predictions
comparison["loaded_prediction"] = loaded_predictions
comparison

In [ ]:
assert np.array_equal(example_predictions, loaded_predictions)
print("Check passed: the logged model reproduces the original predictions.")

### Loading a run's parameters and metrics

So far we've loaded the **model artifact** itself. MLflow also lets you load back a run's **metadata** - its parameters, metrics, and tags - without touching the model at all. This is useful when you want to inspect or compare runs from code rather than clicking through the UI (for example, to build an automated check before deployment).

We can use `MlflowClient` to fetch the run by its `run_id`, then read its data.

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()
run = client.get_run(run_id)

print("Parameters:")
for key, value in run.data.params.items():
    print(f"  {key}: {value}")

print("\nMetrics:")
for key, value in run.data.metrics.items():
    print(f"  {key}: {value:.4f}")

print("\nTags:")
for key, value in run.data.tags.items():
    if not key.startswith("mlflow."):  # skip MLflow's internal tags
        print(f"  {key}: {value}")

# 6. Serve the model as a local endpoint

Loading the model inside Python is useful, but another application should not need direct access to this notebook.

MLflow can expose the model through an HTTP server:

**client application -> POST /invocations -> MLflow model server -> prediction response**

The next cell starts the server on port `5001`.


In [ ]:
MODEL_PORT = 5001
MODEL_BASE_URL = f"http://127.0.0.1:{MODEL_PORT}"

if port_is_open(MODEL_PORT):
    raise RuntimeError(
        f"Port {MODEL_PORT} is already in use. Stop the existing process or change MODEL_PORT."
    )

venv_bin = str(Path(sys.executable).parent)

env = os.environ.copy()
env["MLFLOW_TRACKING_URI"] = TRACKING_URI
env["PATH"] = venv_bin + os.pathsep + env.get("PATH", "")

serve_command = [
    sys.executable,
    "-m",
    "mlflow",
    "models",
    "serve",
    "--model-uri",
    model_uri,
    "--host",
    "127.0.0.1",
    "--port",
    str(MODEL_PORT),
    "--env-manager",
    "local",
]

serve_process, serve_log = start_background_process(
    serve_command,
    "mlflow-model-server.log",
    env=env,
)

if wait_for_url(f"{MODEL_BASE_URL}/health", timeout=60):
    print(f"Model server is healthy at {MODEL_BASE_URL}")
else:
    raise RuntimeError(f"Model server did not start. Inspect the log: {serve_log}")

**Note:** visiting `http://127.0.0.1:5001` directly in a browser will show `{"detail":"Not Found"}`. MLflow's model server doesn't
serve anything at the root path (`/`); it only responds on `/health` (a simple `GET`) and `/invocations` (the prediction endpoint, which requires a `POST` with a JSON body — not something a browser can do by just visiting a
URL).

`/health` returns an empty response body with status `200` — a blank page in the browser is expected there too, not an error. If you want to see the actual status code:

```python
import requests
r = requests.get(f"{MODEL_BASE_URL}/health")
print(r.status_code, repr(r.text))
```

You should see `200 ''`. The real output shows up in the next step, when we send an actual prediction request to `/invocations`.

# 7. Send a valid prediction request

The payload uses the `dataframe_split` format so that column names and order are explicit.


In [ ]:
request_rows = X_test.iloc[:2].copy()
valid_payload = {
    "dataframe_split": request_rows.to_dict(orient="split")
}

response = requests.post(
    f"{MODEL_BASE_URL}/invocations",
    json=valid_payload,
    timeout=10,
)

print(f"Status code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

A successful response demonstrates that:

- The model artifact can be loaded outside the notebook process.
- The endpoint accepts structured input.
- The model signature can validate the request.
- The endpoint returns a machine-readable prediction.


# 8. Test an invalid request

Production systems must fail clearly when inputs do not match the model contract.

This request deliberately removes the `age` feature.


In [ ]:
invalid_rows = request_rows.drop(columns=["age"])
invalid_payload = {
    "dataframe_split": invalid_rows.to_dict(orient="split")
}

invalid_response = requests.post(
    f"{MODEL_BASE_URL}/invocations",
    json=invalid_payload,
    timeout=10,
)

print(f"Status code: {invalid_response.status_code}")
print(invalid_response.text[:1_500])


## Why deliberately test failures?

A production endpoint must do more than return correct predictions for perfect inputs. It should also:

- Reject missing or malformed fields.
- Return useful error information.
- Record failures for investigation.
- Prevent silent use of incorrect data.


# 9. What would we monitor in production?

This local endpoint demonstrates deployment, but it does not implement full production monitoring.

A real system would monitor three layers:

| Layer | Example signals |
|---|---|
| Service health | Availability, error rate, response time, request volume |
| Data health | Missing fields, unexpected values, changes in input distributions |
| Model health | Accuracy or proxy metrics, drift, bias, business outcomes |

It would also define ownership, alert thresholds, rollback procedures, and retraining triggers.


That's it!  In this walkthrough, we took a fitted `scikit-learn` pipeline beyond the notebook: we tracked its parameters, metrics, and lineage in MLflow; packaged it as a portable artifact with a signature and input example; reloaded it independently to confirm it still worked; and served it locally as an HTTP endpoint that accepted valid requests and rejected invalid ones.

# Bring It Home

Complete the sentence:

> A model is ready to leave the notebook when...

Then identify:

1. One thing MLflow tracked.
2. One thing the model signature protected.
3. One thing you would monitor after deployment.
4. One reason you might roll back to an earlier version.


## Optional cleanup

Background processes started by this notebook will be stopped when the kernel shuts down. To stop them immediately, run:


In [ ]:
cleanup_background_processes()
print("Background MLflow processes stopped.")